# Temperature colormap comparison — `Spectral_r` vs `turbo` vs `thermal`

The ECMWF/earthkit `temperature_2m` palette is **`Spectral_r`**, which has a pale, low-saturation midpoint
(≈ +8 °C). On a summer field that pale band lands on the average temperature and **washes the middle of the
map out**. Two colormaps without a pale centre are compared on the *same real ERA5 data and the same scale*:
**`turbo`** (matplotlib) and **`thermal`** (cmocean, via cleopatra's `sea_surface_temperature` style).

In [ ]:
%matplotlib inline
import os, sys
_wt = r"C:/python-environments/worktrees/cleopatra/perceptual-palettes/src"
if os.path.isdir(_wt) and _wt not in sys.path:
    sys.path.insert(0, _wt)
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
import cleopatra
from cleopatra.array_glyph import ArrayGlyph, FrameLabel
from cleopatra.colors import resolve_single_layer_style
from cleopatra.animation import embed_gif
print("cleopatra", cleopatra.__version__)

z = np.load(Path("../data/europe_t2m.npz"), allow_pickle=True)
celsius = z["celsius"].astype(float); labels = list(z["labels"])
ext = [float(v) for v in z["extent"]]
VMIN, VMAX = float(np.nanmin(celsius)), float(np.nanmax(celsius))   # same scale for all three

CMAPS = {
    "Spectral_r (ECMWF/earthkit)": mpl.colormaps["Spectral_r"],
    "turbo": mpl.colormaps["turbo"],
    "thermal (cmocean)": resolve_single_layer_style("sea_surface_temperature")[1]["cmap"],
}
print("scale:", (round(VMIN, 1), round(VMAX, 1)), "degC")

## Side by side — the warmest day, same scale

`Spectral_r`'s pale midpoint washes central Europe; `turbo` and `thermal` keep contrast across the whole field.

In [ ]:
day = int(np.argmax([np.nanmean(f) for f in celsius]))
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), constrained_layout=True)
for ax, (name, cm) in zip(axes, CMAPS.items()):
    ArrayGlyph(celsius[day], extent=[ext[0], ext[2], ext[1], ext[3]], ax=ax).plot(
        cmap=cm, vmin=VMIN, vmax=VMAX, title=name, title_size=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"ERA5 2 m temperature — {labels[day]} (warmest day)", fontweight="bold")
plt.show()

## Animation — `turbo`

In [ ]:
def animate_with(cmap, prefix):
    # figure sized to the data box so the equal-aspect map fills it (no letterbox)
    w, h = ext[1] - ext[0], ext[3] - ext[2]
    fs = (8, 8 * h / w) if w >= h else (8 * w / h, 8)
    glyph = ArrayGlyph(celsius, extent=[ext[0], ext[2], ext[1], ext[3]], figsize=fs)
    anim = glyph.animate(labels, cmap=cmap, vmin=VMIN, vmax=VMAX, title=prefix,
                         cbar_label="2 m temperature (°C)",
                         frame_label=FrameLabel(location=[ext[0] + 0.03 * (ext[1] - ext[0]),
                                                          ext[2] + 0.05 * (ext[3] - ext[2])]),
                         interval=140)
    plt.close(glyph.fig)
    return embed_gif(anim, fps=7)

animate_with(CMAPS["turbo"], "ERA5 2 m temperature (turbo)")

## Animation — `thermal`

In [ ]:
animate_with(CMAPS["thermal (cmocean)"], "ERA5 2 m temperature (thermal)")